# Modeling
---

## 3. Build the preprocessing pipeline

Models need fully numeric, comparable inputs. A `ColumnTransformer` prepares
every column in one object:

- numeric columns (the questions + `age`): fill missing values, then scale
- categorical columns (`gender`, `hand`): fill missing values, then one-hot encode

We keep the target as **text** (e.g. `"Resilient"`). scikit-learn models handle
string labels fine, and it means our predictions come out readable — no label
encoder needed.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Features (X) and target (y). y stays as text labels.
X = df.drop(columns=["target"])
y = df["target"]

numeric_features = item_cols + ["age"]
categorical_features = ["gender", "hand"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

print("Data and preprocessor ready. Shape:", df.shape)

### What the preprocessor actually does

In [ ]:
X_prepared = preprocessor.fit_transform(X)
print("Before:", X.shape, " After:", X_prepared.shape)
# Notice the column count changes — gender/hand became several 0/1 columns.

In [ ]:
# Which columns come out of the preprocessor?
feature_names = preprocessor.get_feature_names_out()
print(list(feature_names[-8:]))

# First rows after preprocessing: scaled numbers + 0/1 columns, and no gaps left
prepared_df = pd.DataFrame(X_prepared, columns=feature_names)
print("Missing values after preprocessing:", int(prepared_df.isna().sum().sum()))
prepared_df.head().round(2)

In [ ]:
model

## 4. Plug a model into the pipeline

We glue the preprocessor in front of a model, so the whole thing is **one object**. Here we first check the random forest model on a simple train/test split and read the **classification report**, which scores each class separately.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print("Accuracy on the test set:", round(accuracy_score(y_test, predictions), 3))
print()
print("A few predictions:", list(predictions[:5]))
print("Actual labels:    ", list(y_test[:5]))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

The line we care about is **`macro avg` → `f1-score`**.

Macro F1 averages the F1 of every class **equally**, so a small class counts as much as a big one. That matters here because our classes are imbalanced (look
back at the target counts in LS1). From now on, **macro F1 is our single number** for comparing models.

### Is 0.84 accuracy actually good? A baseline for comparison

A "model" that always predicts the most common class shows how much accuracy we get for free.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])
baseline.fit(X_train, y_train)
print("Baseline accuracy (always 'Moderate'):", round(accuracy_score(y_test, baseline.predict(X_test)), 3))

In [ ]:
# Accuracy per class: how often is each type predicted correctly?
per_class = (
    pd.DataFrame({"actual": y_test, "correct": predictions == y_test})
    .groupby("actual")["correct"].mean()
    .round(3)
    .sort_values()
)
per_class

Since Undercontroller is defined only by C and A, which only had one question each represented out of the 19 from dataset. This explains directly why the model only catches ~20% of Undercontrollers while getting most of the Resilients.

## Think about — answers

**Why might accuracy be misleading here?**
The classes are imbalanced. Always predicting "Moderate" already gives ~43 % accuracy without learning anything.
The overall accuracy of ~0.85 also hides big differences between the classes: the model recognises
Resilient and Moderate very well, but only about a quarter of the **Undercontrollers**. Because this is
the smallest class, it barely affects the overall accuracy. That is why Live Session 2 uses F1-macro,
which treats every class equally.

**What happens if a new user leaves a question blank?**
The pipeline handles it: the `SimpleImputer` fills a blank question with the median and a blank
gender/hand with the most frequent value, so the model still returns a prediction (tested above).
The prediction is based on a guessed value, though, so in the app it is better to make every answer required.

**RandomForest vs. LogisticRegression?**
Both reach about the same accuracy (~0.85), LogisticRegression is even slightly higher. The simpler,
linear model is enough here, probably because the answers relate to the types in a fairly straightforward
way (higher N-scores → more likely Overcontroller, lower → Resilient). A single split is not enough to pick
a winner — we compare them properly with cross-validation and F1-macro in Live Session 2.

## Recap & what's next

We loaded the data, looked at it, built a preprocessing **pipeline**, and
confirmed a model runs end-to-end through it.

In **Live Session 2** we evaluate models properly with cross-validation,
compare two of them, and tune them.

👉 **Your turn:** plug a different model into the pipeline (e.g.
`LogisticRegression`) and see whether the accuracy changes.

### Trying LogisticRegression instead of RandomForest

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

log_reg_model.fit(X_train, y_train)
log_reg_predictions = log_reg_model.predict(X_test)

print("RandomForest accuracy:      ", round(accuracy_score(y_test, predictions), 3))
print("LogisticRegression accuracy:", round(accuracy_score(y_test, log_reg_predictions), 3))

# Imports

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/data_cleaned.csv")

In [ ]:
df

In [ ]:
X = df.drop(columns="target")

In [ ]:
y = df["target"]

# Pipelines

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, make_column_selector(dtype_include='number')),
        ('cat', categorical_transformer, ['gender', 'hand'])
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

In [ ]:
pipeline

# Cross Validation